# logit-thermo 探索：場の下見の h ＝ 0 を同じ種で回し直し、各歩の「候補の確率の和 q」を記録する（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 35〜40 分。終わったら自動で切れます。
- 探索（予想なし）。8.30 の場の下見の h ＝ 0（Pythia-1B・OLMo-2-1B、温度 0.4〜0.7、サンプル 4、記事 40）と**同じ種**で生成し、トークンが 8.30 と 1 つ残らず同じことを確かめてから、各歩の q（候補の確率の和、float32）も保存します。q があれば、h ＝ 0 の生成だけから場の効きを厳密に予言でき、8.30 の h ＝ ±0.3 とくらべられます。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコードと `scripts/a2_explore_field.py`（`0ac42c7ab82b06d2…`）は Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_explore_fq.py`（`8f734f990df61e07…`）は埋め込み。
- 保存先 `A2_explore/fq/`。8.30 のデータ（`A2_explore/field/`）を読みます。途中で切れても、もう一度「すべて実行」すれば終わったモデルは飛ばします。**結果は表示しません。**
- **通知**：🚀・モデルごとの ✅・🎉・⚠️。名札は「logit-thermo 探索 q の記録（場の下見の h ＝ 0 を同じ種で回し直し、Pythia-1B・OLMo-1B、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 q の記録（場の下見の h ＝ 0 を同じ種で回し直し、Pythia-1B・OLMo-1B、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore q record', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF']
SRCQ = '"""探索 8.35：場の下見（8.30）の h ＝ 0 を同じ種でもう一度生成し、各歩の「候補の確率の和 q」を記録する（2026-10-31 14:3x、GPU（L4）、判定しない）。\nscripts/a2_2_collect.py・a2_4_collect.py・a2_7_collect.py・a2_7b_collect.py・a2_explore_field.py を実行したあとの同じ名前空間で実行する\n（load_model・valid_mask・split_doc・CopyIndexN・B7_N・CONT・F_TEMPS・F_S・f_seed・generate_field を変えずに使う）。\n\n使い道：厳密な重みづけ直しと一次の応答（docs/A2_EQUATION_CATALOG.md の式 24・31）。1 歩ずつ割り直す生成では\n  d⟨X⟩/dh（h ＝ 0）＝ Cov(X, Σ_t（候補を選んだか_t − q_t）) ÷ T\nが厳密に成り立つので、h ＝ 0 の生成だけから場の効きを予言し、8.30 の h ＝ ±0.3 の生成とくらべる。\n記録：トークン、候補があったか、候補を選んだか、q（その歩の、場をかけた後の候補の確率の和）、q0（場をかける前の、同じ温度での候補の確率の和）。\n  h ＝ 0 では q ＝ q0。確率の計算は倍精度、保存は float32（1 の近くの刻み 6 × 10^−8）。\nトークンが 8.30 の h ＝ 0（Drive の A2_explore/field/{名前}@h0.npz）と 1 つ残らず同じことを確かめてから保存する。\n"""\nimport gc, os, time\nimport numpy as np\nimport torch\n\nQ_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\')]\n\n\n@torch.no_grad()\ndef generate_fq(model, prompts, T, valid, gen, h):\n    """generate_field と同じ手順（同じ乱数の使い方）で、q・q0 も記録する。"""\n    device = valid.device\n    B = len(prompts)\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    idx = [CopyIndexN(p, B7_N) for p in prompts]\n    toks, cand, pick, Q, Q0 = [], [], [], [], []\n    for t in range(CONT):\n        z = logits.double().masked_fill(~valid, float(\'-inf\'))\n        cands = [idx[b].candidates() for b in range(B)]\n        has = torch.tensor([len(c) > 0 for c in cands], device=device)\n        rows, cols = [], []\n        for b, c in enumerate(cands):\n            rows += [b] * len(c); cols += list(c)\n        rt = torch.tensor(rows, device=device, dtype=torch.long); ct = torch.tensor(cols, device=device, dtype=torch.long)\n        q0 = torch.zeros(B, dtype=torch.float64, device=device)\n        if rows:\n            q0.index_add_(0, rt, torch.softmax(z / T, dim=-1)[rt, ct])\n        if h != 0.0 and rows:\n            z[rt, ct] += h\n        p = torch.softmax(z / T, dim=-1)\n        q = torch.zeros(B, dtype=torch.float64, device=device)\n        if rows:\n            q.index_add_(0, rt, p[rt, ct])\n        nxt = torch.multinomial(p, 1, generator=gen)\n        pk = torch.tensor([int(nxt[b, 0]) in set(cands[b]) for b in range(B)], device=device)\n        for b in range(B):\n            idx[b].push(int(nxt[b, 0]))\n        toks.append(nxt[:, 0]); cand.append(has); pick.append(pk); Q.append(q); Q0.append(q0)\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    st = lambda v: torch.stack(v, 1).cpu().numpy()\n    return dict(toks=st(toks).astype(np.int32), cand=st(cand).astype(np.bool_), pick=st(pick).astype(np.bool_),\n                q=st(Q).astype(np.float32), q0=st(Q0).astype(np.float32))\n\n\ndef run_fq(name, hf_id, docs, outdir, fielddir, device, notify=print):\n    path = f\'{outdir}/{name}@h0q.npz\'\n    if os.path.exists(path):\n        print(name, \'はもう終わっています（飛ばします）\')\n        return\n    ref = np.load(f\'{fielddir}/{name}@h0.npz\')\n    assert float(ref[\'h\']) == 0.0\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, eos = valid_mask(tok, model, device)\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    assert use == np.asarray(ref[\'use\']).tolist(), \'記事の組が 8.30 と違います\'\n    nD = len(use)\n    prompts = [pairs[i][0] for i in use]\n    R = {k: np.zeros((len(F_TEMPS), F_S, nD, CONT), dt)\n         for k, dt in ((\'toks\', np.int32), (\'cand\', np.bool_), (\'pick\', np.bool_), (\'q\', np.float32), (\'q0\', np.float32))}\n    for k, T in enumerate(F_TEMPS):\n        for s in range(F_S):\n            g = torch.Generator(device=device).manual_seed(f_seed(k, s))\n            r = generate_fq(model, prompts, T, valid, g, 0.0)\n            for kk in R:\n                R[kk][k, s] = r[kk]\n        same = bool((R[\'toks\'][k] == ref[\'toks\'][k]).all() and (R[\'pick\'][k] == ref[\'pick\'][k]).all())\n        assert same, (name, \'温度\', T, \'のトークンが 8.30 の h ＝ 0 と違います\')\n    np.savez_compressed(path + \'.part.npz\', temps=np.array(F_TEMPS), h=0.0, use=np.array(use), seconds=time.time() - t0, **R)\n    os.replace(path + \'.part.npz\', path)\n    notify(f\'{name}@h0q おわり（8.30 とトークンが一致、{time.time() - t0:.0f} 秒）\')\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\n@torch.no_grad()\ndef self_test_fq(device):\n    """小さなランダムのモデルで：トークンは generate_field と同じ（h ＝ 0・＋0.3）。q は全部を読み直して出した値と合う。h ＝ 0 では q ＝ q0。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    global CONT\n    keep = CONT\n    CONT = 60\n    try:\n        prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4, [3, 4, 3, 4] * 4]\n        for h in (0.0, 0.3):\n            a = generate_field(model, prompts, 0.6, valid, torch.Generator(device=device).manual_seed(11), h)\n            b = generate_fq(model, prompts, 0.6, valid, torch.Generator(device=device).manual_seed(11), h)\n            assert (a[\'toks\'] == b[\'toks\']).all() and (a[\'pick\'] == b[\'pick\']).all(), (\'generate_field と違う\', h)\n            assert ((b[\'q\'] == 0) | b[\'cand\']).all() and ((b[\'q0\'] == 0) | b[\'cand\']).all()\n            assert (b[\'q\'] >= 0).all() and (b[\'q\'] <= 1 + 1e-6).all()\n            if h == 0.0:\n                assert np.array_equal(b[\'q\'], b[\'q0\'])\n            else:\n                assert (b[\'q\'][b[\'cand\']] >= b[\'q0\'][b[\'cand\']] - 1e-6).all()\n            # 読み直し：1 本目の文で、各歩の候補を CopyIndexN でなぞり、q を出し直す\n            seq = b[\'toks\'][0].tolist()\n            x = torch.tensor([prompts[0] + seq], device=device)\n            lg = model(x).logits[0, len(prompts[0]) - 1: len(prompts[0]) - 1 + 60].double().masked_fill(~valid, float(\'-inf\'))\n            ci = CopyIndexN(prompts[0], B7_N)\n            for t in range(60):\n                c = list(ci.candidates())\n                zz = lg[t].clone()\n                if c and h != 0.0:\n                    zz[c] += h\n                pp = torch.softmax(zz / 0.6, -1)\n                qq = float(pp[c].sum()) if c else 0.0\n                assert abs(qq - float(b[\'q\'][0, t])) < 2e-3, (h, t, qq, float(b[\'q\'][0, t]))\n                ci.push(seq[t])\n    finally:\n        CONT = keep\n    return \'OK\'\n'
assert hashlib.sha256(SRCQ.encode()).hexdigest() == '8f734f990df61e072a1ef417094e50ca65120b183418ea96d5966337d5994659'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCQ)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 2 つのモデル → 8.30 とトークンが同じか確かめて保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_fq('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore/fq', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    for name, hf_id in Q_MODELS:
        run_fq(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_explore/fq', '/content/drive/MyDrive/logit-thermo/A2_explore/field', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(models=Q_MODELS, temps=F_TEMPS, samples=F_S, seed=F_SEED, field_sha256='0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', fq_sha256='8f734f990df61e072a1ef417094e50ca65120b183418ea96d5966337d5994659',
                   torch=torch.__version__, transformers=transformers.__version__, python=sys.version.split()[0],
                   gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_explore/fq' + '/run_meta.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 q の記録で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：q の記録（8.30 の h ＝ 0 と同じトークン）を A2_explore/fq に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
